# Celdas Voronoi con K-Means esférico

1. Lee `train.parquet` (solo latitud y longitud).
2. Ajusta K-Means esférico con 70 clusters sobre el train; los centroides definen las celdas de Voronoi esféricas.
3. Asigna cada fila de train y de validación a la celda del centroide más cercano (distancia angular).
4. Guarda `train_celdas.parquet` y `val_celdas.parquet` con todas las columnas originales más `ind_cluster`, y `centroides.csv`.

Los parquets se procesan por lotes para no cargar las imágenes en memoria.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from tqdm.auto import tqdm

RUTA_SAMPLE = "datasets/samples/muestra_n-20000_umbral-csv-1000_balanceado_paises-71_seed-42_val-20pct_por-panoid"
K = 70
N_INIT = 5
MAX_ITER = 100
TOL = 1e-7
SEMILLA = 42
COL_LAT, COL_LON = "latitude", "longitude"
TAMANO_LOTE = 512

raiz = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file()), None)
if raiz is None:
    raise FileNotFoundError("Ejecutá el notebook dentro del proyecto.")
carpeta = Path(RUTA_SAMPLE).expanduser()
carpeta = (carpeta if carpeta.is_absolute() else raiz / carpeta).resolve() / "data"
ruta_train, ruta_val = carpeta / "train.parquet", carpeta / "validation.parquet"
salida_train, salida_val = carpeta / "train_celdas.parquet", carpeta / "val_celdas.parquet"
for r in (ruta_train, ruta_val):
    if not r.is_file():
        raise FileNotFoundError(r)
print(carpeta)

C:\Users\juanm\Documents\GitHub\geo-project\datasets\samples\muestra_n-20000_umbral-csv-1000_balanceado_paises-71_seed-42_val-20pct_por-panoid\data


## 1. Coordenadas del train

In [2]:
coords = pq.read_table(ruta_train, columns=[COL_LAT, COL_LON]).to_pandas()
coords.columns = ["lat", "lon"]
n_invalidas = (~(np.isfinite(coords.lat) & np.isfinite(coords.lon)
                 & coords.lat.between(-90, 90) & coords.lon.between(-180, 180))).sum()
if n_invalidas:
    raise ValueError(f"{n_invalidas} filas con coordenadas inválidas.")
print(f"{len(coords):,} filas de train")
coords.describe()

16,018 filas de train


,lat,lon
count,16018.000000,16018.000000
mean,25.840366,21.623534
std,29.610143,62.255488
min,-54.606129,-150.261645
25%,6.663079,-3.593589
50%,38.504188,18.017038
75%,49.024588,56.732809
max,71.012238,178.278816


## 2. K-Means esférico

Cada punto se convierte a un vector unitario en $\mathbb{R}^3$. La asignación es por mayor producto punto
(menor distancia angular) y el centroide es la media de los vectores reproyectada a la esfera.
Inicialización tipo k-means++ con distancia angular.

In [3]:
def to_xyz(lat, lon):
    lat, lon = np.radians(lat), np.radians(lon)
    return np.column_stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)])

def to_latlon(xyz):
    xyz = xyz / np.linalg.norm(xyz, axis=1, keepdims=True)
    return np.degrees(np.arcsin(np.clip(xyz[:, 2], -1, 1))), np.degrees(np.arctan2(xyz[:, 1], xyz[:, 0]))

def asignar(X, C, chunk=200_000):
    labels = np.empty(len(X), dtype=np.int32)
    sim = np.empty(len(X))
    for i in range(0, len(X), chunk):
        s = X[i:i + chunk] @ C.T
        labels[i:i + chunk] = s.argmax(axis=1)
        sim[i:i + chunk] = s.max(axis=1)
    return labels, sim

def init_pp(X, k, rng):
    C = [X[rng.integers(len(X))]]
    d = 1 - np.clip(X @ C[0], -1, 1)
    for _ in range(1, k):
        C.append(X[rng.choice(len(X), p=d / d.sum())])
        d = np.minimum(d, 1 - np.clip(X @ C[-1], -1, 1))
    return np.array(C)

def spherical_kmeans(X, k, n_init=5, max_iter=100, tol=1e-7, seed=0):
    rng = np.random.default_rng(seed)
    best = None
    for _ in range(n_init):
        C = init_pp(X, k, rng)
        prev = -np.inf
        for _ in range(max_iter):
            labels, sim = asignar(X, C)
            for j in range(k):
                m = labels == j
                if m.any():
                    v = X[m].sum(axis=0)
                    C[j] = v / np.linalg.norm(v)
                else:  # cluster vacío: reubicar en el punto peor representado
                    C[j] = X[sim.argmin()]
            score = sim.sum()
            if score - prev < tol * len(X):
                break
            prev = score
        labels, sim = asignar(X, C)
        if best is None or sim.sum() > best[0]:
            best = (sim.sum(), C.copy())
    return best[1]

X_train = to_xyz(coords.lat.to_numpy(), coords.lon.to_numpy())
centroides = spherical_kmeans(X_train, K, N_INIT, MAX_ITER, TOL, SEMILLA)
labels_train, sim = asignar(X_train, centroides)
dist_km = np.arccos(np.clip(sim, -1, 1)) * 6371.0
print(f"celdas no vacías: {len(np.unique(labels_train))} de {K}")
print(f"distancia al centroide (km): media {dist_km.mean():.0f}, máx {dist_km.max():.0f}")

celdas no vacías: 70 de 70
distancia al centroide (km): media 275, máx 2563


In [4]:
cent_lat, cent_lon = to_latlon(centroides)
tabla_cent = pd.DataFrame({"ind_cluster": range(K), "lat": cent_lat, "lon": cent_lon,
                           "n_train": np.bincount(labels_train, minlength=K)})
tabla_cent.to_csv(carpeta / "centroides.csv", index=False)
tabla_cent.sort_values("n_train", ascending=False).head(10)

,ind_cluster,lat,lon,n_train
22,22,45.566424,16.777294,1024
40,40,49.542788,17.440027,928
0,0,56.711233,24.587006,714
44,44,51.320546,5.218630,646
30,30,57.649444,11.062594,564
67,67,44.101583,24.187042,524
10,10,46.411054,6.634821,500
29,29,40.251008,-6.143447,441
55,55,53.257798,-5.581280,434
21,21,25.117702,88.367413,417


### Mapa del diagrama de Voronoi

In [5]:
import plotly.graph_objects as go
from scipy.spatial import SphericalVoronoi

MAX_PUNTOS_MAPA = 30_000  # solo afecta al dibujo

def arco(a, b, n=24):
    omega = np.arccos(np.clip(a @ b, -1, 1))
    if omega < 1e-9:
        return np.array([a, b])
    t = np.linspace(0, 1, n)[:, None]
    return (np.sin((1 - t) * omega) * a + np.sin(t * omega) * b) / np.sin(omega)

def aristas_voronoi(C):
    sv = SphericalVoronoi(C, radius=1.0)
    sv.sort_vertices_of_regions()
    vistas, lats, lons = set(), [], []
    for region in sv.regions:
        for i in range(len(region)):
            u, v = region[i], region[(i + 1) % len(region)]
            clave = (min(u, v), max(u, v))
            if clave in vistas:
                continue
            vistas.add(clave)
            la, lo = to_latlon(arco(sv.vertices[u], sv.vertices[v]))
            la, lo = list(la), list(lo)
            # cortar la línea si cruza el antimeridiano
            for j in reversed(np.where(np.abs(np.diff(lo)) > 180)[0]):
                la.insert(j + 1, None); lo.insert(j + 1, None)
            lats += la + [None]; lons += lo + [None]
    return lats, lons

e_lat, e_lon = aristas_voronoi(centroides)
muestra = np.random.default_rng(SEMILLA).choice(len(coords), min(MAX_PUNTOS_MAPA, len(coords)), replace=False)

fig = go.Figure()
fig.add_trace(go.Scattergeo(lat=coords.lat.to_numpy()[muestra], lon=coords.lon.to_numpy()[muestra], mode="markers",
                            marker=dict(size=3, color=labels_train[muestra], colorscale="Turbo", opacity=0.6),
                            hoverinfo="skip", name="train"))
fig.add_trace(go.Scattergeo(lat=e_lat, lon=e_lon, mode="lines", line=dict(width=1.5, color="#d62728"), name="Voronoi"))
fig.add_trace(go.Scattergeo(lat=cent_lat, lon=cent_lon, mode="markers+text", text=[str(i) for i in range(K)],
                            textposition="top center", marker=dict(size=8, color="black", symbol="x"), name="centroides"))
fig.update_geos(projection_type="natural earth", showland=True, landcolor="#f1ede4", showocean=True,
                oceancolor="#dfeaf2", showcountries=True, countrycolor="#bbbbbb", showcoastlines=True, coastlinecolor="#888888")
fig.update_layout(title=f"Diagrama de Voronoi esférico (k={K})", height=650, margin=dict(l=0, r=0, t=50, b=0),
                  legend=dict(x=0.01, y=0.01))
fig.show()

In [6]:
# Exportar lat/lon de cada prototipo (centroide) de cluster
ruta_prototipos = carpeta / "prototipos.csv"
pd.DataFrame({"ind_cluster": range(K), "lat": cent_lat, "lon": cent_lon}).to_csv(ruta_prototipos, index=False)
print(f"{K} prototipos guardados en {ruta_prototipos}")

70 prototipos guardados en C:\Users\juanm\Documents\GitHub\geo-project\datasets\samples\muestra_n-20000_umbral-csv-1000_balanceado_paises-71_seed-42_val-20pct_por-panoid\data\prototipos.csv


## 3. Guardar `train_celdas` y `val_celdas`

Se copian todas las columnas y se agrega `ind_cluster` (int32) con el índice de la celda de Voronoi.

In [7]:
def agregar_celdas(origen, destino):
    pf = pq.ParquetFile(origen)
    esquema = pf.schema_arrow.append(pa.field("ind_cluster", pa.int32()))
    contador = np.zeros(K, dtype=np.int64)
    destino_tmp = destino.with_suffix(".parquet.tmp")
    try:
        with pq.ParquetWriter(destino_tmp, esquema, compression="snappy") as w:
            for lote in tqdm(pf.iter_batches(batch_size=TAMANO_LOTE), total=-(-pf.metadata.num_rows // TAMANO_LOTE), desc=origen.name):
                lat = lote.column(lote.schema.get_field_index(COL_LAT)).to_numpy(zero_copy_only=False)
                lon = lote.column(lote.schema.get_field_index(COL_LON)).to_numpy(zero_copy_only=False)
                lab, _ = asignar(to_xyz(lat, lon), centroides)
                contador += np.bincount(lab, minlength=K)
                w.write_batch(lote.append_column("ind_cluster", pa.array(lab, type=pa.int32())))
        destino_tmp.replace(destino)
    finally:
        destino_tmp.unlink(missing_ok=True)
    return contador

n_train = agregar_celdas(ruta_train, salida_train)
n_val = agregar_celdas(ruta_val, salida_val)
assert (n_train == tabla_cent.n_train.to_numpy()).all()
print(f"train_celdas: {n_train.sum():,} filas | val_celdas: {n_val.sum():,} filas")

train.parquet:   0%|          | 0/32 [00:00<?, ?it/s]

validation.parquet:   0%|          | 0/8 [00:00<?, ?it/s]

train_celdas: 16,018 filas | val_celdas: 3,982 filas


## 4. Verificación

In [8]:
for ruta in (salida_train, salida_val):
    pf = pq.ParquetFile(ruta)
    print(ruta.name, pf.metadata.num_rows, "filas;", pf.schema_arrow.names[-3:])
resumen = pd.DataFrame({"train": n_train, "validation": n_val}).rename_axis("ind_cluster")
print(f"celdas sin imágenes en validación: {(resumen.validation == 0).sum()}")
resumen.describe().round(1)

train_celdas.parquet 16018 filas; ['longitude', 'elevation', 'ind_cluster']
val_celdas.parquet 3982 filas; ['longitude', 'elevation', 'ind_cluster']
celdas sin imágenes en validación: 0


,train,validation
count,70.0,70.0
mean,228.8,56.9
std,193.3,47.7
min,32.0,7.0
25%,99.8,25.2
50%,179.5,46.5
75%,263.5,66.8
max,1024.0,257.0
